# Week 65, Day 2: Algorithms vs constants

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w065d2.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
def steps_linear(n):
    return 100 * n          # a slow step, done n times

def steps_quadratic(n):
    return n * n            # a fast step, done n * n times

for n in (10, 100, 1000, 100000):
    winner = "quadratic" if steps_quadratic(n) < steps_linear(n) else "linear"
    print(n, winner)

In [ ]:
def count_pairs_quadratic(items, target):
    ops, found = 0, 0
    for i in range(len(items)):
        for j in range(i + 1, len(items)):
            ops += 1
            if items[i] + items[j] == target:
                found += 1
    return found, ops

def count_pairs_linear(items, target):
    ops, found, seen = 0, 0, {}
    for x in items:
        ops += 1
        found += seen.get(target - x, 0)
        seen[x] = seen.get(x, 0) + 1
    return found, ops

for n in (100, 200, 400):
    data = list(range(n))
    (a, ops_q), (b, ops_l) = count_pairs_quadratic(data, n), count_pairs_linear(data, n)
    print(n, a == b, ops_q, ops_l)

In [ ]:
def insertion_sort(a, counter):
    a = list(a)
    for i in range(1, len(a)):
        j = i
        while j > 0:
            counter[0] += 1
            if a[j - 1] <= a[j]:
                break
            a[j - 1], a[j] = a[j], a[j - 1]
            j -= 1
    return a

def merge_sort(a, counter):
    if len(a) <= 1:
        return list(a)
    mid = len(a) // 2
    left, right = merge_sort(a[:mid], counter), merge_sort(a[mid:], counter)
    out, i, j = [], 0, 0
    while i < len(left) and j < len(right):
        counter[0] += 1
        if left[i] <= right[j]:
            out.append(left[i]); i += 1
        else:
            out.append(right[j]); j += 1
    return out + left[i:] + right[j:]

import random
random.seed(1)
for n in (8, 64, 512):
    data = random.sample(range(n * 10), n)
    c1, c2 = [0], [0]
    insertion_sort(data, c1)
    merge_sort(data, c2)
    print(n, "insertion", c1[0], "merge", c2[0])
nearly = list(range(512))
nearly[100], nearly[101] = nearly[101], nearly[100]
c1, c2 = [0], [0]
insertion_sort(nearly, c1)
merge_sort(nearly, c2)
print("nearly sorted:", c1[0], c2[0])

In [ ]:
def copied_by_concat(pieces):
    text, copied = "", 0
    for p in pieces:
        text = text + p              # builds a new string each time
        copied += len(text)          # every character so far is copied
    return copied

def copied_by_join(pieces):
    return sum(len(p) for p in pieces)   # join copies each piece once

pieces = ["word "] * 1000
print(copied_by_concat(pieces), copied_by_join(pieces))

orders = [{"id": i, "customer": i % 50} for i in range(1000)]
customers = [{"id": c, "name": f"c{c}"} for c in range(50)]
scans = 0
for order in orders:                  # slow: a scan per order
    for c in customers:
        scans += 1
        if c["id"] == order["customer"]:
            break
by_id = {c["id"]: c for c in customers}  # fast: one pass to build, then lookups
names = [by_id[o["customer"]]["name"] for o in orders]
print(scans, len(customers) + len(orders), names[:3])

In [ ]:
import timeit

data = list(range(10000))

def loop_sum():
    total = 0
    for x in data:
        total += x
    return total

def builtin_sum():
    return sum(data)

def loop_squares():
    out = []
    for x in data:
        out.append(x * x)
    return out

def comp_squares():
    return [x * x for x in data]

print(loop_sum() == builtin_sum(), loop_squares() == comp_squares())
ratio = min(timeit.repeat(loop_sum, number=20, repeat=3)) / min(timeit.repeat(builtin_sum, number=20, repeat=3))
print("the built-in is faster:", ratio > 1)

In [ ]:
class Settings:
    def __init__(self):
        self.reads = 0
        self._rate = 1.2
    @property
    def rate(self):
        self.reads += 1
        return self._rate

def scaled_slow(values, settings):
    return [v * settings.rate for v in values]

def scaled_fast(values, settings):
    rate = settings.rate                  # hoisted: read once
    return [v * rate for v in values]

s1, s2 = Settings(), Settings()
print(scaled_slow([1, 2, 3] * 100, s1) == scaled_fast([1, 2, 3] * 100, s2), s1.reads, s2.reads)

> **Key idea.** For large inputs, the growth rate dominates: look first for a better algorithm or data structure, and check growth by measuring at doubling sizes. Constants decide for small inputs, for algorithms with the same growth, and for structured data, which is why hybrids like Timsort switch to insertion sort for small runs. Hunt hidden O(n²): in, index, remove on lists in loops, pop(0), += on strings, repeated sorting or min, and nested scans (build a dict once). In CPython, built-ins and comprehensions run their loops in C or with fewer instructions, locals are faster than globals and attributes, and calls cost; tune only hot code. Hoist loop-invariant work, avoid repeated conversions and I/O, and update derived data incrementally.

### Exercise: Where the lines cross

Write `crossover(cost_a, cost_b, limit=10**9)`: the smallest whole number `n >= 1` at which `cost_a(n) > cost_b(n)` (so from there on, at least for well-behaved costs, `b` is the better choice), or `None` if there isn't one up to `limit`. Search by doubling `n` until `a` costs more, then binary search between the last two values, so it takes about `2 * log2(n)` steps rather than `n`. Assume that once `a` costs more, it keeps costing more. Then `growth(cost, n)`: `cost(2 * n) / cost(n)`, rounded to one decimal place.

In [ ]:
def crossover(cost_a, cost_b, limit=10**9):
    ___

def growth(cost, n):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_crossover():\n    \"comparing growth\"\n    assert crossover(lambda n: n * n, lambda n: 100 * n) == 101\n    assert crossover(lambda n: n * n, lambda n: 5 * n + 50) == 11\n    assert crossover(lambda n: n, lambda n: n * n) is None\n    calls = []\n    def a(n):\n        calls.append(n)\n        return n * n\n    assert crossover(a, lambda n: 1000 * n) == 1001 and len(calls) < 50\n    assert growth(lambda n: n * n, 100) == 4.0 and growth(lambda n: 3 * n, 7) == 2.0\n    assert growth(lambda n: n ** 3, 10) == 8.0")

### Exercise: A hybrid sort

Write `hybrid_sort(items, threshold=16)`: a merge sort that sorts any part of size `threshold` or less with insertion sort instead of splitting further. It returns `(sorted_list, comparisons)`, counting every comparison of two items (in insertion sort, each `a[j - 1] <= a[j]` test; in merging, each `left[i] <= right[j]` test). Don't change the input list. With `threshold=1` it's a plain merge sort.

In [ ]:
def hybrid_sort(items, threshold=16):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_hybrid():\n    \"algorithms and constants\"\n    import random\n    random.seed(7)\n    data = random.sample(range(10000), 500)\n    copy = list(data)\n    out, comps = hybrid_sort(data)\n    assert out == sorted(data) and data == copy\n    _, merge_only = hybrid_sort(data, threshold=1)\n    _, mostly_insertion = hybrid_sort(data, threshold=1000)\n    assert merge_only < mostly_insertion\n    nearly = list(range(2000))\n    nearly[500], nearly[501] = nearly[501], nearly[500]\n    _, a = hybrid_sort(nearly, threshold=1)\n    _, b = hybrid_sort(nearly, threshold=4000)\n    assert b < a and b == 2000\n    assert hybrid_sort([]) == ([], 0) and hybrid_sort([3, 1, 2], threshold=1)[0] == [1, 2, 3]\n    assert hybrid_sort([2, 1], threshold=1) == ([1, 2], 1)")

### Exercise: Finding common items without a scan

Write `common(a, b)`: the items of `a` that also appear in `b`, in `a`'s order, without repeats, in roughly O(len(a) + len(b)) time: the tests count equality comparisons, so `x in b` on a list (which compares `x` with every item) will fail them. Items are hashable. Then `first_repeat(items)`: the first item that has already appeared earlier in the list, or `None`, also in linear time.

In [ ]:
def common(a, b):
    ___

def first_repeat(items):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_membership():\n    \"hidden quadratic code\"\n    class Key:\n        eq_calls = 0\n        def __init__(self, v):\n            self.v = v\n        def __eq__(self, other):\n            Key.eq_calls += 1\n            return isinstance(other, Key) and self.v == other.v\n        def __hash__(self):\n            return hash(self.v)\n        def __repr__(self):\n            return f\"Key({self.v})\"\n    a = [Key(i) for i in range(2000)] + [Key(5)]\n    b = [Key(i) for i in range(1000, 3000)]\n    Key.eq_calls = 0\n    result = common(a, b)\n    assert [k.v for k in result] == list(range(1000, 2000)) and Key.eq_calls < 5000\n    Key.eq_calls = 0\n    items = [Key(i) for i in range(3000)] + [Key(7), Key(9)]\n    assert first_repeat(items).v == 7 and Key.eq_calls < 100\n    assert first_repeat([1, 2, 3]) is None and common([3, 1, 3, 2], [2, 3]) == [3, 2]")

### Exercise: Hoisting repeated work

The tests pass in a `Config` object whose `rate` and `exempt` attributes **count** how often they're read. Write `apply_tax(prices, config)`: a list with each price multiplied by `config.rate` rounded to 2 places, except prices whose item names are in `config.exempt` (a list of names), which are unchanged. `prices` is a list of `(name, price)` pairs. Read each config attribute **at most once**, and make the exemption check a set lookup.

In [ ]:
def apply_tax(prices, config):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_hoist():\n    \"loop-invariant work\"\n    class Config:\n        def __init__(self):\n            self.reads = {\"rate\": 0, \"exempt\": 0}\n        @property\n        def rate(self):\n            self.reads[\"rate\"] += 1\n            return 1.2\n        @property\n        def exempt(self):\n            self.reads[\"exempt\"] += 1\n            return [\"bread\", \"milk\"]\n    c = Config()\n    prices = [(\"tea\", 2.5), (\"bread\", 1.0), (\"jam\", 3.0), (\"milk\", 0.9)] * 250\n    out = apply_tax(prices, c)\n    assert out[:4] == [3.0, 1.0, 3.6, 0.9] and len(out) == 1000\n    assert c.reads == {\"rate\": 1, \"exempt\": 1}\n    assert apply_tax([], Config()) == []")

### Exercise: A queue that doesn't shift

Simulate a ticket system: `serve(arrivals)` processes a list of events, where a string is a new customer joining the back of the queue and `None` means the next customer at the front is served. Return the list of customers in the order served (ignoring `None` when the queue is empty). Use `collections.deque`: the tests check that you don't use `pop(0)` or `insert(0, ...)`. Then `last_k(events, k)`: the last `k` items of an iterable (a list), using `deque(maxlen=k)`, in order.

In [ ]:
from collections import deque

def serve(arrivals):
    ___

def last_k(events, k):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_queue():\n    \"the right structure\"\n    assert serve([\"ada\", \"ben\", None, \"cy\", None, None, None]) == [\"ada\", \"ben\", \"cy\"]\n    assert serve([None, \"x\", None]) == [\"x\"]\n    big = [\"c%d\" % i for i in range(50000)] + [None] * 50000\n    assert serve(big)[-1] == \"c49999\"\n    assert last_k(range(10), 3) == [7, 8, 9] and last_k([1], 5) == [1]\n    assert \"pop(0)\" not in source and \"insert(0\" not in source and \"deque(\" in source")

### Bug hunt: Faster, but out of order

Someone replaced a slow de-duplication loop with `sorted(set(...))`, which is fast but puts the names in alphabetical order instead of the order they first appeared, so this prints `['ada', 'ben', 'zara']`. Use `dict.fromkeys`, which is just as fast and keeps first occurrences in order, so this prints `['zara', 'ada', 'ben']`.

In [ ]:
names = ["zara", "ada", "zara", "ben", "ada"]
unique = sorted(set(names))
print(unique)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"fast and correct\"\n    expect_output(\"['zara', 'ada', 'ben']\")")

### Bug hunt: Hoisted too far

To save work, the discount was moved out of the loop, but it depends on each item's quantity, so every item gets the first item's discount: this prints `[18.0, 9.0, 27.0]`. Keep the loop-invariant part (the rate) outside and compute each item's discount inside, so it prints `[18.0, 10.0, 27.0]`.

In [ ]:
rate = 0.1
items = [(20.0, 5), (10.0, 1), (30.0, 4)]
discount = rate if items[0][1] >= 3 else 0
prices = [round(price * (1 - discount), 2) for price, qty in items]
print(prices)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"invariant work only\"\n    expect_output(\"[18.0, 10.0, 27.0]\")")

### Bug hunt: Binary search on unsorted data

The linear search was "optimised" into a binary search with `bisect`, but the data isn't sorted, so the search looks in the wrong half and this prints `False`. Sort the data once before searching (keeping the binary search) so it prints `True`.

In [ ]:
import bisect

data = [42, 7, 19, 3, 88, 51]

def contains(sorted_items, x):
    i = bisect.bisect_left(sorted_items, x)
    return i < len(sorted_items) and sorted_items[i] == x

print(contains(data, 3))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"sorted first\"\n    expect_output(\"True\")")

### Puzzle: Building a lookup once

Arrange the lines of a function that finds each order's customer name with a dict built once, instead of scanning the customers for every order. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # the smallest n at which n * n exceeds 10 * n
q = ___   # if doubling n multiplies the work by 4, the growth is "linear", "quadratic" or "cubic"?
r = ___   # True or False: x in some_list takes about the same time however long the list is
s = ___   # characters copied building "a" * 4 by four += of "a", counting each full copy: 1 + 2 + 3 + 4
t = ___   # the collections type for a queue with fast removal from the front

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 11 and q == \"quadratic\" and r is False, \"p, q and r\"\n    assert s == 10 and t == \"deque\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: removing the hidden loops

Here's a slow report, correct but full of hidden O(n²) work:

```
def slow_report(orders, customers):
    lines = ""
    names = []
    for order in orders:
        for c in customers:
            if c["id"] == order["customer"]:
                name = c["name"]
        if name not in names:
            names.append(name)
        lines += f"{order['id']}:{name}\n"
    totals = []
    for name in names:
        totals.append((name, sum(o["amount"] for o in orders
                                 if [c for c in customers if c["id"] == o["customer"]][0]["name"] == name)))
    return lines, names, totals
```

Write `fast_report(orders, customers)` returning exactly the same three results (the order lines as one string, the customer names in order of first order, and `(name, total amount)` pairs in that same order), but making **one pass** over `customers` and **one pass** over `orders`. The tests wrap the lists in objects that count how many times each is iterated over, and check that results match `slow_report` on random data. Build strings with `join`.

In [ ]:
def fast_report(orders, customers):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_fast_report():\n    \"one pass each\"\n    import random\n    def slow_report(orders, customers):\n        lines = \"\"\n        names = []\n        for order in orders:\n            for c in customers:\n                if c[\"id\"] == order[\"customer\"]:\n                    name = c[\"name\"]\n            if name not in names:\n                names.append(name)\n            lines += f\"{order['id']}:{name}\\n\"\n        totals = []\n        for name in names:\n            totals.append((name, sum(o[\"amount\"] for o in orders\n                                     if [c for c in customers if c[\"id\"] == o[\"customer\"]][0][\"name\"] == name)))\n        return lines, names, totals\n    class Counted(list):\n        def __init__(self, items):\n            super().__init__(items)\n            self.passes = 0\n        def __iter__(self):\n            self.passes += 1\n            return super().__iter__()\n    random.seed(3)\n    customers = [{\"id\": i, \"name\": f\"cust{i}\"} for i in random.sample(range(1000), 60)]\n    orders = [{\"id\": n, \"customer\": random.choice(customers)[\"id\"], \"amount\": random.randint(1, 50)} for n in range(400)]\n    expected = slow_report(orders, customers)\n    co, cc = Counted(orders), Counted(customers)\n    assert fast_report(co, cc) == expected\n    assert co.passes <= 1 and cc.passes <= 1\n    assert fast_report([], [{\"id\": 1, \"name\": \"x\"}]) == (\"\", [], [])\n    assert \"join\" in source")

## Warm-up

For small inputs, the quadratic method's small constant wins. Past n = 100, the linear one wins, and by n = 100,000 it does a thousand times less work. That's the whole story of today in four lines: constants matter for small inputs, growth matters for large ones.

## Growth beats constants

Big-O (Week 17) describes how the work **grows**, ignoring constant factors: an O(n) algorithm doing 100 steps per item and an O(n²) one doing 1 step per pair. For any constants, there's a size beyond which the slower-growing algorithm wins, and beyond it the gap grows without limit:

| n | 100 n | n² | n log₂ n |
|---|---|---|---|
| 10 | 1,000 | 100 | 33 |
| 1,000 | 100,000 | 1,000,000 | 9,966 |
| 1,000,000 | 10⁸ | 10¹² | 2 x 10⁷ |

At a million items, 10¹² simple operations take hours in Python; 2 x 10⁷ take a second or two. So when a program is slow on large inputs, **first look for a better algorithm or data structure**: a set instead of a list for membership, a dict instead of a nested search, sorting once instead of searching repeatedly, a heap instead of repeated `min` (Phase 4). Constant-factor tuning comes second.

To check the growth of your own code, measure (or count) at a few sizes, each double the last: if the work doubles, it's linear; if it quadruples, quadratic.

## When constants win

Constant factors decide in three situations:

- **The input is small.** For a handful of items, a simple loop beats a clever structure. That's why Python's own `sorted` (Timsort) sorts short runs with **insertion sort** (O(n²) but very fast for tiny n) before merging them, and why many libraries switch algorithms below a threshold.
- **The algorithms have the same growth.** Two O(n) solutions differ only in constants, so the one doing less work per item, or doing it in C, wins.
- **The input has structure.** Insertion sort is O(n) on data that's already nearly sorted; Timsort exploits existing sorted runs. "Worst case" isn't always the case you have.

A **hybrid** algorithm uses the best tool at each size: merge sort for large ranges, switching to insertion sort below a cut-off chosen by measurement.

## Hidden quadratic code

The most common performance bug is O(n²) work hiding behind a line that **looks** like one step:

- **`x in some_list`** inside a loop: `in` on a list scans it. Use a set or dict (O(1) on average, Week 44).
- **`list.index`, `list.count`, `list.remove`** in a loop: each is a scan.
- **`list.pop(0)` or `list.insert(0, x)`** in a loop: every call shifts the whole list. Use `collections.deque`.
- **Building a string with `+=` in a loop**: each concatenation may copy everything so far. CPython sometimes avoids the copy, but don't rely on it: collect pieces in a list and `"".join(pieces)` once.
- **Repeated sorting**, or `min`/`max` of a growing list on each step: sort once, or use a heap (`heapq`).
- **Nested searches**: for each order, search all customers for the matching one. Build a dict from id to customer once, then look up.

## Python's constant factors

In CPython, each line of Python code costs dozens of bytecode instructions (Week 64), while built-in functions run their loops in C. So for the same algorithm:

- **Built-ins beat hand-written loops**: `sum(xs)`, `max(xs)`, `sorted(xs)`, `"".join(parts)`, `set(xs)`, `any(...)`, `dict.fromkeys(xs)`, `str.count`, slicing. Push the loop into C whenever there's a built-in for it.
- **Comprehensions beat `for` loops with `append`**: no `.append` lookup and method call per item.
- **Local variables beat globals and attributes**: `LOAD_FAST` versus dict lookups. In a very hot loop, `append = out.append` or binding a global to a local helps, a little.
- **Fewer function calls**: each Python call has overhead (a frame, argument handling). Inlining a tiny helper in a hot loop, or using `map` with a built-in, saves time.
- **The right types**: `int` arithmetic is fast; `Decimal` and `Fraction` are much slower (but exact); `bytes` operations beat processing characters one by one.

These matter only in **hot** code, the lines a profile shows running millions of times. Elsewhere, write the clearest version.

## Avoiding repeated work

The cheapest operation is one you don't do. Look for:

- **Loop-invariant work**: something computed on every pass that doesn't change, such as `len(x)`, `config.rate`, a compiled regular expression, `math.sqrt(2)`, a lower-cased search term. Compute it once, before the loop (**hoisting**).
- **Repeated conversions**: turning the same list into a set inside a function that's called many times; parsing the same date string again and again.
- **Recomputing derived data**: totals recalculated from scratch after every small change, where updating them incrementally would do (a running total, Week 41's prefix sums).
- **Repeated I/O**: reading a file or querying a database in a loop, instead of once.

Thursday adds the general tool for repeated work: **caching**.

## Practice

## Quiz

## Daily challenge

## Before you go

Find one loop in your own code that contains `in` on a list, `.index`, `.pop(0)` or `+=` on a string, and decide whether it could ever run on large inputs. Tomorrow: choosing fast data structures from the standard library.